**Fig. 4B and Video S2: whole-tissue run on the scratch-wounded epidermis (first time point) with the DSM pair**

Needs a GPU.

In [ ]:
# Working directory: data/fig4/ under the repository root, where the Fig. 4 deposit folder is downloaded
# (domain-specialized_finetuned_bootstrap/ = the DSM model pair, one folder per sample with its
# input volume and the final mask, wound_finetuning/).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig4")
print("working directory:", os.getcwd())


# EpiCycle ScaledSegmentation Pipeline

Run 3D cell segmentation, reconstruction, and assembly on tiled image data.

**Usage:** Fill in the paths and parameters below, then Run All.

## 1. Paths & Input

In [ ]:
# ===== PATHS (edit these) =====

# Input Z-stack TIFF (the original microscope file name is in the README) — set this to tile from scratch, or leave None if already tiled
INPUT_TIF = "wound_timepoint1/input.tif"   # Set to None if tiles already exist

# Working directory (tiles + outputs go here)
# If INPUT_TIF is set, tiles will be created at WORK_DIR/tiles/
# If INPUT_TIF is None, WORK_DIR should contain existing tile subdirectories
WORK_DIR = "wound_timepoint1"

# CellPose model paths (.pth files)
LATERAL_MODEL = "domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootstrap4Finetuned_lateral_500epoch_scratch_lenient_rescale_True"
AXIAL_MODEL = "domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootrap4Finetuned_axial_500epoch_scratch_lenient_rescale_True"

# Output directory name (created inside WORK_DIR)
OUTPUT_NAME = "Wound_timepoint1"

## 2. Resource Configuration

In [ ]:
# ===== RESOURCES (auto-detected from the job's allocation if None) =====

N_GPUS = None            # None = auto-detect from SLURM/CUDA
N_CPU_PER_WORKER = None  # None = auto (total_cores / n_gpus in parallel mode)

## 3. Stage Selection & Tiling Parameters

In [ ]:
# Which stages to run (set False to skip)
RUN_TILING = INPUT_TIF is not None   # Auto: tile if input TIFF provided
# RUN_TILING = False
RUN_SEGMENTATION = True
RUN_RECONSTRUCTION = True
RUN_ASSEMBLY = True 
RUN_GLOBAL_RECONSTRUCTION = True

# ===== TILING (Stage 0) =====
PATCH_SIZE = 1024           # Target XY patch size in pixels
# OVERLAP is shared with assembly (set in Section 6)
OVERLAP = 0.5

Z_THRESHOLD_METHOD = "hard"  # "hard", "percentile", or "default"
Z_THRESHOLD_VALUE = 0.07     # Intensity threshold for Z-range detection
CHANNEL_INDEX = 1            # Which channel for Z-range analysis
AUTO_CORRECT_PATCH = False   # Auto-find valid patch size if exact tiling fails

## 4. Segmentation Parameters

In [ ]:
# ===== SEGMENTATION (Stage 1) =====
# Recorded for reference: the stage is built without segment_params below, so the engine
# defaults were used (channels [2, 1], cell probability 0, flow threshold 0.2). The flow
# threshold acts on the 2D masks only; the 3D reconstruction uses the probability and flow maps.

# GPU allocation strategy: "auto", "fixed" (3-GPU triplets), "round_robin"
GPU_STRATEGY = "auto"

# CellPose parameters
FLOW_THRESHOLD = 0.4
CELLPROB_THRESHOLD = 0.0
CHANNELS = [2, 1]      # [signal, nucleus] — adjust for your data
BATCH_SIZE = 1024

## 5. Reconstruction Parameters

In [ ]:
# ===== RECONSTRUCTION (Stage 2) =====

# Per-tile direct reconstruction: the engine's u-Segment3D defaults, with gradient renormalisation
# off as in the engine version the runs used (EpiCycle v0.1.2 defaults to on).
RECON_RENORM_GRADIENT = False

# "sequential" (one tile at a time, all cores) or "parallel" (multiple tiles)
RECON_MODE = "parallel"
# RECON_MODE = "sequential"

# Max concurrent tiles in parallel mode (None = n_gpus)
# Rule of thumb: 1900 TB node / ~450 GB per 1920x1920 tile ≈ 4 max
RECON_MAX_WORKERS = None

# Post-processing
SAVE_PROB3D = True
SAVE_FLOW3D = True
SAVE_BINARY = True
BINARY_ERODE = True
BINARY_KERNEL_SIZE = 3
BINARY_BORDER_MARGIN = 2
BINARY_FILL_HOLES = True

# Delete stage1 intermediates after reconstruction to save disk
CLEANUP_INTERMEDIATES = False

## 6. Assembly Parameters

In [ ]:
# ===== ASSEMBLY (Stage 3) =====

# Full volume dimensions in pixels (H, W)
FULL_SIZE = (4096, 4096)

# Tile size and overlap (must match how tiles were cut)
TILE_SIZE = PATCH_SIZE

# Number of Z slices (None = auto-detect from first tile)
DEPTH = None

# Which volumes to stitch
STITCH_MASK = True
STITCH_PROB = True
STITCH_BINARY = True

# Overlap blending for probabilities: "average" or "max"
PROB_STITCH_MODE = "max"

## 7. Global Indirect Reconstruction Parameters

In [ ]:
# ===== GLOBAL INDIRECT RECONSTRUCTION (Stage 4) =====

DOWNSAMPLE_FACTOR = 2          # Downsample stitched volume by this factor
GLOBAL_MIN_SIZE = 1000         # Remove objects smaller than this (voxels, after downsampling)
USE_PRECOMPUTED_BINARY = True  # Use stitched binary mask as foreground constraint

# Gradient descent parameters
# GLOBAL_N_ITER = 30            # Gradient descent iterations (more = finer, slower)
GLOBAL_N_ITER = 300            # Gradient descent iterations (more = finer, slower)
GLOBAL_MOMENTA = 0.9          # Momentum for gradient descent
# GLOBAL_GRADIENT_DECAY = 0.001    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.01    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.1    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.05    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
GLOBAL_GRADIENT_DECAY = 0.25    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower

GLOBAL_RENORM_GRADIENT = True  # Renormalize gradient at each step

# Indirect method parameters
# GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method edt is simple but there maybe artifacts in the seems
GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method edt is simple but there maybe artifacts in the seems
GLOBAL_SMOOTH_SKEL_SIGMA = 1   # Skeleton smoothing sigma
GLOBAL_N_CPU = 64              # CPU threads for multiprocess watershed
GLOBAL_TILE_SHAPE = None       # Tile shape for watershed (None = auto)

## 8. Setup & Pre-flight Check

In [ ]:
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

# Resolve GPU count
from packages.epicycle_scaledseg.utils.resources import get_allocated_gpus
n_gpus = N_GPUS or get_allocated_gpus()

work_dir = Path(WORK_DIR)
work_dir.mkdir(parents=True, exist_ok=True)

# Output lives inside the working directory
output_dir = work_dir / OUTPUT_NAME
output_dir.mkdir(exist_ok=True)
log_dir = output_dir / "logs"
log_dir.mkdir(exist_ok=True)

# Tiles directory
tiles_dir = work_dir / "tiles"

if RUN_TILING:
    print(f"Input TIFF: {INPUT_TIF}")
    print(f"Tiles will be created in: {tiles_dir}")
    print(f"Patch size: {PATCH_SIZE}, overlap: {OVERLAP}")
else:
    # Discover existing tile directories
    tile_dirs = sorted([
        str(p) for p in work_dir.iterdir()
        if p.is_dir() and not p.name.startswith(("stage1", "stage2", "stage3", "stage4", "logs", ".", "epicycle"))
        and p.name != OUTPUT_NAME
    ])
    print(f"Found {len(tile_dirs)} existing tile directories in {work_dir}")
    for td in tile_dirs[:5]:
        print(f"  {Path(td).name}")
    if len(tile_dirs) > 5:
        print(f"  ... and {len(tile_dirs) - 5} more")

print(f"\nGPUs: {n_gpus}")
print(f"Output: {output_dir}")

Input TIFF: <run>/WT_WoundHealing_TimePoints/12-2-25 cdh1 h2b - Denoised.nd2 - Stitched-1-01_processed-4096x4096.tif
Tiles will be created in: <run>/WT_WoundHealing_TimePoints/Wound_timepoint1/tiles
Patch size: 1024, overlap: 0.5

GPUs: 3
Output: <run>/WT_WoundHealing_TimePoints/Wound_timepoint1/Wound_timepoint1


## 9. Build & Run Pipeline

In [ ]:
from packages.epicycle_scaledseg.core.pipeline import EpiCyclePipeline

pipeline = EpiCyclePipeline(
    work_dir=str(work_dir),
    n_gpus=n_gpus,
    log_dir=str(log_dir),
)

# --- Stage 0: Tiling ---
if RUN_TILING:
    from packages.epicycle_scaledseg.core.tiling import TilingStage
    pipeline.add_stage(TilingStage(
        input_tif=INPUT_TIF,
        output_dir=str(tiles_dir),
        patch_size=PATCH_SIZE,
        overlap=OVERLAP,
        z_threshold_method=Z_THRESHOLD_METHOD,
        threshold_value=Z_THRESHOLD_VALUE,
        channel_index=CHANNEL_INDEX,
        auto_correct=AUTO_CORRECT_PATCH,
    ))
    # Tiling creates tile_dirs; use placeholder for now
    tile_dirs = [str(work_dir)]

# --- Stage 1: Segmentation ---
if RUN_SEGMENTATION:
    from packages.epicycle_scaledseg.core.segmentation import SegmentationStage
    pipeline.add_stage(SegmentationStage(
        lateral_model=LATERAL_MODEL,
        axial_model=AXIAL_MODEL,
        out_root=str(output_dir / "stage1_segmentation"),
        gpu_strategy=GPU_STRATEGY,
    ))

# --- Stage 2: Reconstruction ---
if RUN_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.reconstruction import ReconstructionStage
    from packages.epicycle_scaledseg.config.defaults import get_default_reconstruct_params
    reconstruct_params = get_default_reconstruct_params()
    reconstruct_params['gradient_descent']['renorm_gradient'] = RECON_RENORM_GRADIENT
    pipeline.add_stage(ReconstructionStage(
        seg_root=str(output_dir / "stage1_segmentation"),
        out_root=str(output_dir / "stage2_reconstruction"),
        mode=RECON_MODE,
        reconstruct_params=reconstruct_params,
        max_workers=RECON_MAX_WORKERS,
        n_cpu_per_worker=N_CPU_PER_WORKER,
        save_prob3d=SAVE_PROB3D,
        save_flow3d=SAVE_FLOW3D,
        save_binary=SAVE_BINARY,
        binary_erode=BINARY_ERODE,
        binary_kernel_size=BINARY_KERNEL_SIZE,
        binary_border_margin=BINARY_BORDER_MARGIN,
        binary_fill_holes=BINARY_FILL_HOLES,
        cleanup_intermediates=CLEANUP_INTERMEDIATES,
    ))

# --- Stage 3: Assembly ---
if RUN_ASSEMBLY:
    from packages.epicycle_scaledseg.core.assembly import AssemblyStage
    pipeline.add_stage(AssemblyStage(
        recon_root=str(output_dir / "stage2_reconstruction"),
        output_dir=str(output_dir / "stage3_assembly"),
        full_size=FULL_SIZE,
        tile_size=TILE_SIZE,
        overlap=OVERLAP,
        depth=DEPTH,
        stitch_mask=STITCH_MASK,
        stitch_prob=STITCH_PROB and SAVE_PROB3D,
        stitch_binary=STITCH_BINARY and SAVE_BINARY,
        prob_stitch_mode=PROB_STITCH_MODE,
    ))

# --- Stage 4: Global Indirect Reconstruction ---
if RUN_GLOBAL_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.global_reconstruction import GlobalReconstructionStage
    from packages.epicycle_scaledseg.config.defaults import get_default_indirect_reconstruct_params

    indirect_params = get_default_indirect_reconstruct_params(n_cpu=GLOBAL_N_CPU)
    indirect_params['gradient_descent']['n_iter'] = GLOBAL_N_ITER
    indirect_params['gradient_descent']['momenta'] = GLOBAL_MOMENTA
    indirect_params['gradient_descent']['gradient_decay'] = GLOBAL_GRADIENT_DECAY
    indirect_params['gradient_descent']['renorm_gradient'] = GLOBAL_RENORM_GRADIENT
    indirect_params['indirect_method']['dtform_method'] = GLOBAL_DTFORM_METHOD
    indirect_params['indirect_method']['smooth_skel_sigma'] = GLOBAL_SMOOTH_SKEL_SIGMA
    if GLOBAL_TILE_SHAPE is not None:
        indirect_params['gradient_descent']['tile_shape'] = GLOBAL_TILE_SHAPE

    pipeline.add_stage(GlobalReconstructionStage(
        assembly_dir=str(output_dir / "stage3_assembly"),
        output_dir=str(output_dir / "stage4_global"),
        downsample_factor=DOWNSAMPLE_FACTOR,
        min_size=GLOBAL_MIN_SIZE,
        use_precomputed_binary=USE_PRECOMPUTED_BINARY,
        indirect_params=indirect_params,
        n_cpu=GLOBAL_N_CPU,
        tile_shape=GLOBAL_TILE_SHAPE,
    ))

print(f"Pipeline: {len(pipeline.stages)} stages")
for s in pipeline.stages:
    print(f"  - {s.name}")
print(f"Tile dirs: {len(tile_dirs)}")
print(f"Output: {output_dir}")

# ===== RUN =====
result = pipeline.run(tile_dirs)
print()
print(result.summary())

2026-02-03 19:19:30,762 epicycle.pipeline INFO Pipeline starting: 5 stage(s), 1 tile(s), 3 GPU(s)
2026-02-03 19:19:30,762 epicycle.pipeline INFO Run parameters saved to data/WT_WoundHealing_TimePoints/Wound_timepoint1/Wound_timepoint1/logs/run_params.json
2026-02-03 19:19:30,762 epicycle.pipeline INFO === Stage: tiling ===
2026-02-03 19:19:30,763 epicycle.tiling INFO Stage 'tiling' starting with 1 tile(s)
2026-02-03 19:19:30,764 epicycle.tiling INFO Parameters: {"input_tif": "data/WT_WoundHealing_TimePoints/12-2-25 cdh1 h2b - Denoised.nd2 - Stitched-1-01_processed-4096x4096.tif", "patch_size": 1024, "overlap": 0.5, "z_threshold_method": "hard", "threshold_value": 0.07, "channel_index": 1, "auto_correct": false}
2026-02-03 19:19:30,764 epicycle.core.tiling INFO Tiling data/WT_WoundHealing_TimePoints/12-2-25 cdh1 h2b - Denoised.nd2 - Stitched-1-01_processed-4096x4096.tif -> data/WT_WoundHealing_TimePoints/Wound_timepoint1/tiles (patch=1024, overlap=0.5)
2026-02-03 19:19:30,765 epicycle.u

Pipeline: 5 stages
  - tiling
  - segmentation
  - reconstruction
  - assembly
  - global_reconstruction
Tile dirs: 1
Output: <run>/WT_WoundHealing_TimePoints/Wound_timepoint1/Wound_timepoint1


2026-02-03 19:19:58,643 epicycle.utils.tiling INFO Z-stack shape: Z=572, C=2, H=4096, W=4096, dtype=uint16
2026-02-03 19:19:58,647 epicycle.utils.tiling INFO Tiling: patch_size=1024, step=512, overlap=0.5
2026-02-03 19:19:58,648 epicycle.utils.tiling INFO Grid: 7 rows x 7 cols = 49 tiles
Saving tiles: 100%|██████████████████████████████████████████| 49/49 [01:06<00:00,  1.35s/tile]
2026-02-03 19:21:04,981 epicycle.core.tiling INFO Tiling complete: 49 tiles, actual patch=1024, step=512
2026-02-03 19:21:05,050 epicycle.pipeline INFO Stage 'tiling': 49/49 succeeded (94.3s)
2026-02-03 19:21:05,051 epicycle.tiling INFO Stage 'tiling' finished: 49/49 succeeded (94.3s)
2026-02-03 19:21:05,051 epicycle.pipeline INFO Stage 'tiling' produced 49 tile directories
2026-02-03 19:21:05,052 epicycle.pipeline INFO === Stage: segmentation ===
2026-02-03 19:21:05,053 epicycle.segmentation INFO Stage 'segmentation' starting with 49 tile(s)
2026-02-03 19:21:05,053 epicycle.segmentation INFO Parameters: {"l

Automatric binary thresholding:  0.7
[ 371 1216 1216] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  144
start dynamics
Tile id 30 72689 72689
Tile id 24 153970 153970
Tile id 18 198717 198717
Tile id 31 217870 217870
Tile id 32 210980 210980
Tile id 12 295991 295991
Tile id 33 320287 320287
Tile id 35 465980 465980
Tile id 25 488866 488866
Tile id 26 540810 540810
Tile id 6 578442 578442
Tile id 19 702209 702209
Tile id 0 715830 715830
Tile id 29 782431 782431
Tile id 20 809642 809642
Tile id 5 806651 806651
Tile id 27 840315 840315
Tile id 34 845437 845437
Tile id 21 987750 987750
Tile id 23 1012591 1012591
Tile id 2 1101751 1101751
Tile id 1 1179572 1179572
Tile id 4 1123807 1123807
Tile id 13 1225934 1225934
Tile id 7 1245633 1245633
Tile id 14 1296779 1296779
Tile id 3 1360596 1360596
Tile id 22 1372732 1372732
Tile id 28 1350014 1350014
Tile id 17 1419137 1419137
Tile id 36 1474877 1474877
Tile id 41 1477966 1477966
Tile id 72 755207 755207
Tile id 11 1519397 15193

Eroding (GPU): 100%|██████████| 1072/1072 [00:03<00:00, 287.38inst/s]


Automatric binary thresholding:  0.6
[ 399 1216 1216] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  144
start dynamics
Tile id 17 0 0
Tile id 29 0 0
Tile id 35 0 0
Tile id 23 0 0
Tile id 34 0 0
Tile id 28 1696 1696
Tile id 33 5581 5581
Tile id 5 12539 12539
Tile id 32 13716 13716
Tile id 11 30919 30919
Tile id 27 89595 89595
Tile id 22 212136 212136
Tile id 26 283067 283067
Tile id 41 331275 331275
Tile id 16 462933 462933
Tile id 47 487244 487244
Tile id 31 561579 561579
Tile id 59 621614 621614
Tile id 21 665618 665618
Tile id 0 677372 677372
Tile id 53 707322 707322
Tile id 4 746762 746762
Tile id 25 724793 724793
Tile id 71 530405 530405
Tile id 72 557674 557674
Tile id 65 723002 723002
Tile id 30 812472 812472
Tile id 10 809427 809427
Tile id 19 830637 830637
Tile id 20 877563 877563
Tile id 6 1048433 1048433
Tile id 77 361073 361073
Tile id 13 1106703 1106703
Tile id 18 1102662 1102662
Tile id 14 1091046 1091046
Tile id 24 1084324 1084324
Tile id 12 1153492 115349

... (94 outputs left out)

Eroding (GPU): 100%|██████████| 907/907 [00:02<00:00, 398.97inst/s] 
2026-02-03 23:44:42,916 epicycle.pipeline INFO Stage 'reconstruction': 49/49 succeeded (8460.2s)
2026-02-03 23:44:42,917 epicycle.reconstruction INFO Stage 'reconstruction' finished: 49/49 succeeded (8460.2s)
2026-02-03 23:44:42,919 epicycle.pipeline INFO === Stage: assembly ===
2026-02-03 23:44:42,921 epicycle.assembly INFO Stage 'assembly' starting with 49 tile(s)
2026-02-03 23:44:42,921 epicycle.assembly INFO Parameters: {"full_size": [4096, 4096], "tile_size": 1024, "overlap": 0.5, "depth": null, "stitch_mask": true, "stitch_prob": true, "stitch_binary": true, "prob_stitch_mode": "max", "binary_format": "npy", "n_workers": 16}
2026-02-03 23:44:42,922 epicycle.assembly INFO Loading tiling config from data/WT_WoundHealing_TimePoints/Wound_timepoint1/tiles/tiling_config.json
2026-02-03 23:44:42,923 epicycle.assembly INFO Loading tiling config from data/WT_WoundHealing_TimePoints/Wound_timepoint1/tiles/tiling_config.j

[ 382 2240 2240] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  576
start dynamics
Tile id 138 0 0
Tile id 135 0 0
Tile id 90 0 0
Tile id 63 0 0
Tile id 99 0 0
Tile id 54 0 0
Tile id 102 0 0
Tile id 78 0 0
Tile id 123 0 0
Tile id 75 0 0
Tile id 66 0 0
Tile id 87 0 0
Tile id 114 0 0
Tile id 126 0 0
Tile id 111 0 0
Tile id 51 89 89
Tile id 139 0 0
Tile id 136 0 0
Tile id 117 690 690
Tile id 137 0 0
Tile id 91 0 0
Tile id 103 0 0
Tile id 124 0 0
Tile id 79 0 0
Tile id 42 1370 1370
Tile id 76 0 0
Tile id 64 0 0
Tile id 100 0 0
Tile id 67 0 0
Tile id 88 0 0
Tile id 112 0 0
Tile id 52 37 37
Tile id 115 227 227
Tile id 127 227 227
Tile id 81 2472 2472
Tile id 140 1622 1622
Tile id 125 0 0
Tile id 118 192 192
Tile id 101 0 0
Tile id 80 0 0
Tile id 77 0 0
Tile id 65 0 0
Tile id 89 0 0
Tile id 68 0 0
Tile id 113 0 0
Tile id 55 1698 1698
Tile id 53 0 0
Tile id 116 0 0
Tile id 69 11009 11009
Tile id 39 11705 11705
Tile id 119 19 19
Tile id 141 40524 40524
Tile id 132 47698 47698
Til

2026-02-04 00:26:56,953 epicycle.core.global_reconstruction INFO Global reconstruction complete: shape=(286, 2048, 2048), n_labels=15457, elapsed=1998.7s
2026-02-04 00:26:56,955 epicycle.core.global_reconstruction INFO Saving outputs to <run>/WT_WoundHealing_TimePoints/Wound_timepoint1/Wound_timepoint1/stage4_global
2026-02-04 00:28:57,872 epicycle.pipeline INFO Stage 'global_reconstruction': 1/1 succeeded (2165.8s)
2026-02-04 00:28:57,874 epicycle.global_reconstruction INFO Stage 'global_reconstruction' finished: 1/1 succeeded (2165.8s)
2026-02-04 00:28:57,874 epicycle.pipeline WARNING No tiles remaining — stopping pipeline.
2026-02-04 00:28:57,875 epicycle.pipeline INFO Pipeline: 1/49 tiles (18567.1s total)
  tiling: 49/49 succeeded (94.3s)
  segmentation: 49/49 succeeded (7357.7s)
  reconstruction: 49/49 succeeded (8460.2s)
  assembly: 49/49 succeeded (489.2s)
  global_reconstruction: 1/1 succeeded (2165.8s)



Pipeline: 1/49 tiles (18567.1s total)
  tiling: 49/49 succeeded (94.3s)
  segmentation: 49/49 succeeded (7357.7s)
  reconstruction: 49/49 succeeded (8460.2s)
  assembly: 49/49 succeeded (489.2s)
  global_reconstruction: 1/1 succeeded (2165.8s)


## 10. Check Status

In [ ]:
# Quick status check — how many tiles completed each stage?
seg_root = output_dir / "stage1_segmentation"
recon_root = output_dir / "stage2_reconstruction"

seg_done = sum(1 for td in tile_dirs if (seg_root / Path(td).name / "SEGMENTATION_DONE").exists())
recon_done = sum(1 for td in tile_dirs if (recon_root / Path(td).name / f"{Path(td).name}_mask.tif").exists())

print(f"Stage 1 (Segmentation):   {seg_done}/{len(tile_dirs)} complete")
print(f"Stage 2 (Reconstruction): {recon_done}/{len(tile_dirs)} complete")
print(f"Stage 3 (Assembly):       check {output_dir / 'stage3_assembly'}")
print(f"Stage 4 (Global Recon):   {'done' if (output_dir / 'stage4_global' / 'global_segmentation.tif').exists() else 'not done'}")

Stage 1 (Segmentation):   0/1 complete
Stage 2 (Reconstruction): 0/1 complete
Stage 3 (Assembly):       check <run>/WT_WoundHealing_TimePoints/Wound_timepoint1/Wound_timepoint1/stage3_assembly
Stage 4 (Global Recon):   done
